# GraphCast weight entropy on the real mesh lengths

The published checkpoints are not retrained. Each one stores a single mesh-edge encoder and applies it to every multimesh edge, so there is no separate weight matrix for M0, M1, and so on.

The energy at level m is the mean squared output of those shared weights on the edges whose length is L(m). L(m) = pi R / (2 * 2^m), with R = 6371 km, and the wavenumber is k = 1/L. The hierarchy stops at `model_config.mesh_size`: M0-M5 for GraphCast Small, M0-M6 for the operational and full checkpoints. The 50 km, 25 km, and 12 km rungs are not edges of these checkpoints, so they are not given an energy.

Three outputs of the same weights are reported, in nats:

- `linear_0`: the first matrix only, which is the quantity in `Main codes`.
- `mlp`: both linear maps, with swish between them, before layer norm.
- `encoder`: that result after the trained layer norm, which is the edge embedding the encoder writes.

H = -sum p_i ln p_i and H_n = H / ln N.


In [ ]:
import base64
from pathlib import Path
PAYLOAD = {'icosahedral_mesh.py': '...', 'weight_entropy.py': 'IiIiV2VpZ2h0IGVuZXJneSBvZiBHcmFwaENhc3Qgb24gdGhlIG1lc2ggbGVuZ3RocyB0aGUgY2hlY2twb2ludCBhY3R1YWxseSBoYXMuCgpHcmFwaENhc3Qgc3RvcmVzIG9uZSBtZXNoLWVkZ2UgZW5jb2RlciBhbmQgYXBwbGllcyBpdCB0byBldmVyeSBtdWx0aW1lc2ggZWRnZS4KVGhlcmUgaXMgbm8gc2VwYXJhdGUgd2VpZ2h0IG1hdHJpeCBmb3IgTTAsIE0xLCAuLi4gVGhlIGVuZXJneSBhdCBsZXZlbCBtIGlzIHRoZQptZWFuIHNxdWFyZWQgb3V0cHV0IG9mIHRob3NlIHNoYXJlZCB3ZWlnaHRzIG9uIHRoZSBlZGdlcyBvZiB0aGF0IGxldmVsLiBMZXZlbHMKZmluZXIgdGhhbiBtb2RlbF9jb25maWcubWVzaF9zaXplIGFyZSBub3QgaW52ZW50ZWQuCgpIID0gLXN1bSBwX2kgbG4gcF9pIGlzIHJlcG9ydGVkIGluIG5hdHMuIFRoZSB3YXZlbnVtYmVyIGlzIGsgPSAxL0wsIHdpdGggTCB0aGUKcGh5c2ljYWwgbGVuZ3RoIG9mIHRoZSBsZXZlbCBpbiBraWxvbWV0cmVzLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBjc3YKaW1wb3J0IGltcG9ydGxpYi51dGlsCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHVybGxpYi5wYXJzZSBpbXBvcnQgcXVvdGUKZnJvbSB1cmxsaWIucmVxdWVzdCBpbXBvcnQgdXJscmV0cmlldmUKCmltcG9ydCBtYXRwbG90bGliLnB5cGxvdCBhcyBwbHQKaW1wb3J0IG51bXB5IGFzIG5wCmZyb20gc2NpcHkuc3BlY2lhbCBpbXBvcnQgZXhwaXQKZnJvbSBzY2lweS5zdGF0cyBpbXBvcnQgbGlucmVncmVzcwoKUl9LTSA9IDYzNzEuMApQUkVGSVggPSAicGFyYW1zOm1lc2hfZ25uL35fbmV0d29ya3NfYnVpbGRlci9lbmNvZGVyX2VkZ2VzX21lc2giCkJVQ0tFVCA9ICJodHRwczovL3N0b3JhZ2UuZ29vZ2xlYXBpcy5jb20vZG1fZ3JhcGhjYXN0L2dyYXBoY2FzdC9wYXJhbXMvIgoKTU9ERUxTID0gKAogICAgKAogICAgICAgICJzbWFsbCIsCiAgICAgICAgIkdyYXBoQ2FzdF9zbWFsbCAtIEVSQTUgMTk3OS0yMDE1IC0gcmVzb2x1dGlvbiAxLjAgLSBwcmVzc3VyZSBsZXZlbHMgMTMgLSBtZXNoIDJ0bzUgLSBwcmVjaXBpdGF0aW9uIGlucHV0IGFuZCBvdXRwdXQubnB6IiwKICAgICksCiAgICAoCiAgICAgICAgIm9wZXJhdGlvbmFsIiwKICAgICAgICAiR3JhcGhDYXN0X29wZXJhdGlvbmFsIC0gRVJBNS1IUkVTIDE5NzktMjAyMSAtIHJlc29sdXRpb24gMC4yNSAtIHByZXNzdXJlIGxldmVscyAxMyAtIG1lc2ggMnRvNiAtIHByZWNpcGl0YXRpb24gb3V0cHV0IG9ubHkubnB6IiwKICAgICksCiAgICAoCiAgICAgICAgImZ1bGwiLAogICAgICAgICJHcmFwaENhc3QgLSBFUkE1IDE5NzktMjAxNyAtIHJlc29sdXRpb24gMC4yNSAtIHByZXNzdXJlIGxldmVscyAzNyAtIG1lc2ggMnRvNiAtIHByZWNpcGl0YXRpb24gaW5wdXQgYW5kIG91dHB1dC5ucHoiLAogICAgKSwKKQoKCmRlZiBvdXRwdXRfZGlyKCkgLT4gUGF0aDoKICAgIGthZ2dsZSA9IFBhdGgoIi9rYWdnbGUvd29ya2luZyIpCiAgICBpZiBrYWdnbGUuaXNfZGlyKCk6CiAgICAgICAgcmV0dXJuIGthZ2dsZQogICAgcGF0aCA9IFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnQgLyAib3V0cHV0cyIKICAgIHBhdGgubWtkaXIoZXhpc3Rfb2s9VHJ1ZSkKICAgIHJldHVybiBwYXRoCgoKZGVmIGNhY2hlX2RpcigpIC0+IFBhdGg6CiAgICBrYWdnbGUgPSBQYXRoKCIva2FnZ2xlL3dvcmtpbmciKQogICAgaWYga2FnZ2xlLmlzX2RpcigpOgogICAgICAgIHBhdGggPSBrYWdnbGUgLyAiY2hlY2twb2ludHMiCiAgICAgICAgcGF0aC5ta2RpcihleGlzdF9vaz1UcnVlKQogICAgICAgIHJldHVybiBwYXRoCiAgICBwYXRoID0gUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudHNbMV0gLyAiX2NhY2hlIgogICAgcGF0aC5ta2RpcihleGlzdF9vaz1UcnVlKQogICAgcmV0dXJuIHBhdGgKCgpkZWYgbG9hZF9tZXNoKCk6CiAgICBwYXRoID0gUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudCAvICJpY29zYWhlZHJhbF9tZXNoLnB5IgogICAgc3BlYyA9IGltcG9ydGxpYi51dGlsLnNwZWNfZnJvbV9maWxlX2xvY2F0aW9uKCJpY29zYWhlZHJhbF9tZXNoIiwgcGF0aCkKICAgIG1vZHVsZSA9IGltcG9ydGxpYi51dGlsLm1vZHVsZV9mcm9tX3NwZWMoc3BlYykKICAgIHNwZWMubG9hZGVyLmV4ZWNfbW9kdWxlKG1vZHVsZSkKICAgIHJldHVybiBtb2R1bGUKCgpkZWYgY2hlY2twb2ludF9wYXRoKGZpbGVuYW1lOiBzdHIpIC0+IFBhdGg6CiAgICBwYXRoID0gY2FjaGVfZGlyKCkgLyBmaWxlbmFtZQogICAgaWYgbm90IHBhdGguaXNfZmlsZSgpOgogICAgICAgIHByaW50KGYiZG93bmxvYWRpbmcge2ZpbGVuYW1lfSIpCiAgICAgICAgdXJscmV0cmlldmUoQlVDS0VUICsgcXVvdGUoZmlsZW5hbWUpLCBwYXRoKQogICAgcmV0dXJuIHBhdGgKCgpkZWYgZWRnZV9nZW9tZXRyeShtZXNoX21vZHVsZSwgc3BsaXRzOiBpbnQpOgogICAgbWVzaGVzID0gbWVzaF9tb2R1bGUuZ2V0X2hpZXJhcmNoeV9vZl90cmlhbmd1bGFyX21lc2hlc19mb3Jfc3BoZXJlKHNwbGl0cz1zcGxpdHMpCiAgICBtZXJnZWQgPSBtZXNoX21vZHVsZS5tZXJnZV9tZXNoZXMobWVzaGVzKQogICAgc2VuZGVycywgcmVjZWl2ZXJzID0gbWVzaF9tb2R1bGUuZmFjZXNfdG9fZWRnZXMobWVyZ2VkLmZhY2VzKQogICAgZmFjZV9sZXZlbHMgPSBucC5jb25jYXRlbmF0ZSgKICAgICAgICBbCiAgICAgICAgICAgIG5wLmZ1bGwobWVzaC5mYWNlcy5zaGFwZVswXSwgbGV2ZWwsIGR0eXBlPW5wLmludDMyKQogICAgICAgICAgICBmb3IgbGV2ZWwsIG1lc2ggaW4gZW51bWVyYXRlKG1lc2hlcykKICAgICAgICBdCiAgICApCiAgICAjIGZhY2VzX3RvX2VkZ2VzIGdyb3VwcyB0aGUgdGhyZWUgZGlyZWN0ZWQgZWRnZXMgb2YgZXZlcnkgZmFjZSwgc28gdGhlCiAgICAjIGxldmVsIHRhZ3MgYXJlIHRoZSBmYWNlIGxldmVscyByZXBlYXRlZCB0aHJlZSB0aW1lcywgbm90IGludGVybGVhdmVkLgogICAgZWRnZV9sZXZlbHMgPSBucC50aWxlKGZhY2VfbGV2ZWxzLCAzKQogICAgZGlmZmVyZW5jZSA9IG1lcmdlZC52ZXJ0aWNlc1tyZWNlaXZlcnNdIC0gbWVyZ2VkLnZlcnRpY2VzW3NlbmRlcnNdCiAgICBsZW5ndGggPSBucC5saW5hbGcubm9ybShkaWZmZXJlbmNlLCBheGlzPS0xLCBrZWVwZGltcz1UcnVlKQogICAgZmVhdHVyZXMgPSBucC5jb25jYXRlbmF0ZShbZGlmZmVyZW5jZSwgbGVuZ3RoXSwgYXhpcz0tMSkuYXN0eXBlKG5wLmZsb2F0MzIpCiAgICByZXR1cm4gZWRnZV9sZXZlbHMsIGZlYXR1cmVzCgoKZGVmIGVuY29kZXJfb3V0cHV0cyhmZWF0dXJlczogbnAubmRhcnJheSwgY2hlY2twb2ludDogbnAubGliLm5weWlvLk5wekZpbGUpOgogICAgIiIiUmV0dXJuIGxpbmVhcl8wLCB0aGUgc2Vjb25kIGxpbmVhciBtYXAsIGFuZCB0aGUgbGF5ZXItbm9ybSBlbWJlZGRpbmcuIiIiCiAgICBsaW5lYXIwID0gZmVhdHVyZXMgQCBjaGVja3BvaW50W2Yie1BSRUZJWH1fbWxwL34vbGluZWFyXzA6dyJdCiAgICBsaW5lYXIwID0gbGluZWFyMCArIGNoZWNrcG9pbnRbZiJ7UFJFRklYfV9tbHAvfi9saW5lYXJfMDpiIl0KICAgIGhpZGRlbiA9IGxpbmVhcjAgKiBleHBpdChsaW5lYXIwKQogICAgcHJlX25vcm0gPSBoaWRkZW4gQCBjaGVja3BvaW50W2Yie1BSRUZJWH1fbWxwL34vbGluZWFyXzE6dyJdCiAgICBwcmVfbm9ybSA9IHByZV9ub3JtICsgY2hlY2twb2ludFtmIntQUkVGSVh9X21scC9+L2xpbmVhcl8xOmIiXQogICAgbWVhbiA9IHByZV9ub3JtLm1lYW4oYXhpcz0tMSwga2VlcGRpbXM9VHJ1ZSkKICAgIHZhcmlhbmNlID0gcHJlX25vcm0udmFyKGF4aXM9LTEsIGtlZXBkaW1zPVRydWUpCiAgICBub3JtYWxpemVkID0gKHByZV9ub3JtIC0gbWVhbikgLyBucC5zcXJ0KHZhcmlhbmNlICsgMWUtNSkKICAgIGVtYmVkZGluZyA9IG5vcm1hbGl6ZWQgKiBjaGVja3BvaW50W2Yie1BSRUZJWH1fbGF5ZXJfbm9ybTpzY2FsZSJdCiAgICBlbWJlZGRpbmcgPSBlbWJlZGRpbmcgKyBjaGVja3BvaW50W2Yie1BSRUZJWH1fbGF5ZXJfbm9ybTpvZmZzZXQiXQogICAgcmV0dXJuIGxpbmVhcjAsIHByZV9ub3JtLCBlbWJlZGRpbmcKCgpkZWYgbWVhbl9lbmVyZ3kodmFsdWVzOiBucC5uZGFycmF5LCBlZGdlX2xldmVsczogbnAubmRhcnJheSwgbl9sZXZlbHM6IGludCkgLT4gbnAubmRhcnJheToKICAgIGVuZXJneSA9IG5wLmVtcHR5KG5fbGV2ZWxzLCBkdHlwZT1ucC5mbG9hdDY0KQogICAgY291bnRzID0gbnAuZW1wdHkobl9sZXZlbHMsIGR0eXBlPW5wLmludDY0KQogICAgc3F1YXJlZCA9IG5wLnN1bSh2YWx1ZXMuYXN0eXBlKG5wLmZsb2F0NjQpICoqIDIsIGF4aXM9LTEpCiAgICBmb3IgbGV2ZWwgaW4gcmFuZ2Uobl9sZXZlbHMpOgogICAgICAgIG1hc2sgPSBlZGdlX2xldmVscyA9PSBsZXZlbAogICAgICAgIGNvdW50c1tsZXZlbF0gPSBpbnQobWFzay5zdW0oKSkKICAgICAgICBlbmVyZ3lbbGV2ZWxdID0gZmxvYXQoc3F1YXJlZFttYXNrXS5tZWFuKCkpCiAgICByZXR1cm4gZW5lcmd5LCBjb3VudHMKCgpkZWYgZW50cm9weShlbmVyZ3k6IG5wLm5kYXJyYXkpIC0+IHR1cGxlW2Zsb2F0LCBmbG9hdCwgbnAubmRhcnJheV06CiAgICBwcm9iYWJpbGl0eSA9IGVuZXJneSAvIGVuZXJneS5zdW0oKQogICAgc3BlY3RyYWwgPSBmbG9hdCgtbnAuc3VtKHByb2JhYmlsaXR5ICogbnAubG9nKHByb2JhYmlsaXR5KSkpCiAgICByZXR1cm4gc3BlY3RyYWwsIHNwZWN0cmFsIC8gZmxvYXQobnAubG9nKGxlbihlbmVyZ3kpKSksIHByb2JhYmlsaXR5CgoKZGVmIHBvd2VyX2xhdyhlbmVyZ3k6IG5wLm5kYXJyYXksIHdhdmVudW1iZXI6IG5wLm5kYXJyYXkpIC0+IHR1cGxlW2Zsb2F0LCBmbG9hdCwgZmxvYXRdOgogICAgc2xvcGUsIGludGVyY2VwdCwgY29ycmVsYXRpb24sICpfID0gbGlucmVncmVzcyhucC5sb2cod2F2ZW51bWJlciksIG5wLmxvZyhlbmVyZ3kpKQogICAgcmV0dXJuIGZsb2F0KHNsb3BlKSwgZmxvYXQobnAuZXhwKGludGVyY2VwdCkpLCBmbG9hdChjb3JyZWxhdGlvbioqMikKCgpkZWYgYW5hbHl6ZShuYW1lOiBzdHIsIGZpbGVuYW1lOiBzdHIsIG1lc2hfbW9kdWxlKSAtPiB0dXBsZVtsaXN0W2RpY3RdLCBkaWN0XToKICAgIHdpdGggbnAubG9hZChjaGVja3BvaW50X3BhdGgoZmlsZW5hbWUpKSBhcyBjaGVja3BvaW50OgogICAgICAgIHNwbGl0cyA9IGludChjaGVja3BvaW50WyJtb2RlbF9jb25maWc6bWVzaF9zaXplIl0pCiAgICAgICAgaGlkZGVuX2xheWVycyA9IGludChjaGVja3BvaW50WyJtb2RlbF9jb25maWc6aGlkZGVuX2xheWVycyJdKQogICAgICAgIGlmIGhpZGRlbl9sYXllcnMgIT0gMToKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYie25hbWV9OiBleHBlY3RlZCBvbmUgaGlkZGVuIGxheWVyLCBmb3VuZCB7aGlkZGVuX2xheWVyc30iKQogICAgICAgIGVkZ2VfbGV2ZWxzLCBmZWF0dXJlcyA9IGVkZ2VfZ2VvbWV0cnkobWVzaF9tb2R1bGUsIHNwbGl0cykKICAgICAgICBvdXRwdXRzID0gZW5jb2Rlcl9vdXRwdXRzKGZlYXR1cmVzLCBjaGVja3BvaW50KQogICAgICAgIHdlaWdodF9lbmVyZ3kgPSB7CiAgICAgICAgICAgICJsaW5lYXJfMCI6IGZsb2F0KG5wLnN1bShjaGVja3BvaW50W2Yie1BSRUZJWH1fbWxwL34vbGluZWFyXzA6dyJdICoqIDIpKSwKICAgICAgICAgICAgImxpbmVhcl8xIjogZmxvYXQobnAuc3VtKGNoZWNrcG9pbnRbZiJ7UFJFRklYfV9tbHAvfi9saW5lYXJfMTp3Il0gKiogMikpLAogICAgICAgIH0KICAgIG5fbGV2ZWxzID0gc3BsaXRzICsgMQogICAgbGVuZ3RoX2ttID0gbnAuYXJyYXkoW25wLnBpICogUl9LTSAvICgyICogMioqbGV2ZWwpIGZvciBsZXZlbCBpbiByYW5nZShuX2xldmVscyldKQogICAgd2F2ZW51bWJlciA9IDEuMCAvIGxlbmd0aF9rbQogICAgc3RhZ2VfbmFtZXMgPSAoImxpbmVhcl8wIiwgIm1scCIsICJlbmNvZGVyIikKICAgIGVuZXJnaWVzID0ge30KICAgIGNvdW50cyA9IE5vbmUKICAgIGZvciBzdGFnZSwgdmFsdWVzIGluIHppcChzdGFnZV9uYW1lcywgb3V0cHV0cyk6CiAgICAgICAgZW5lcmdpZXNbc3RhZ2VdLCBjb3VudHMgPSBtZWFuX2VuZXJneSh2YWx1ZXMsIGVkZ2VfbGV2ZWxzLCBuX2xldmVscykKICAgICAgICBkZWwgdmFsdWVzCgogICAgcm93cyA9IFtdCiAgICBzdW1tYXJ5ID0gewogICAgICAgICJtb2RlbCI6IG5hbWUsCiAgICAgICAgIm1lc2hfc2l6ZSI6IHNwbGl0cywKICAgICAgICAibl9sZXZlbHMiOiBuX2xldmVscywKICAgICAgICAic3VtX3cyX2xpbmVhcl8wIjogd2VpZ2h0X2VuZXJneVsibGluZWFyXzAiXSwKICAgICAgICAic3VtX3cyX2xpbmVhcl8xIjogd2VpZ2h0X2VuZXJneVsibGluZWFyXzEiXSwKICAgIH0KICAgIHByb2JhYmlsaXRpZXMgPSB7fQogICAgZm9yIHN0YWdlIGluIHN0YWdlX25hbWVzOgogICAgICAgIHNwZWN0cmFsLCBub3JtYWxpemVkLCBwcm9iYWJpbGl0eSA9IGVudHJvcHkoZW5lcmdpZXNbc3RhZ2VdKQogICAgICAgIHNsb3BlLCBwcmVmYWN0b3IsIHIyID0gcG93ZXJfbGF3KGVuZXJnaWVzW3N0YWdlXSwgd2F2ZW51bWJlcikKICAgICAgICBwcm9iYWJpbGl0aWVzW3N0YWdlXSA9IHByb2JhYmlsaXR5CiAgICAgICAgc3VtbWFyeVtmIkhfe3N0YWdlfSJdID0gc3BlY3RyYWwKICAgICAgICBzdW1tYXJ5W2YiSG5fe3N0YWdlfSJdID0gbm9ybWFsaXplZAogICAgICAgIHN1bW1hcnlbZiJzbG9wZV97c3RhZ2V9Il0gPSBzbG9wZQogICAgICAgIHN1bW1hcnlbZiJwcmVmYWN0b3Jfe3N0YWdlfSJdID0gcHJlZmFjdG9yCiAgICAgICAgc3VtbWFyeVtmInIyX3tzdGFnZX0iXSA9IHIyCiAgICAgICAgc3VtbWFyeVtmIm0wX3tzdGFnZX0iXSA9IGZsb2F0KHByb2JhYmlsaXR5WzBdKQogICAgZm9yIGxldmVsIGluIHJhbmdlKG5fbGV2ZWxzKToKICAgICAgICByb3cgPSB7CiAgICAgICAgICAgICJtb2RlbCI6IG5hbWUsCiAgICAgICAgICAgICJsZXZlbCI6IGxldmVsLAogICAgICAgICAgICAiTF9rbSI6IGZsb2F0KGxlbmd0aF9rbVtsZXZlbF0pLAogICAgICAgICAgICAia19wZXJfa20iOiBmbG9hdCh3YXZlbnVtYmVyW2xldmVsXSksCiAgICAgICAgICAgICJuX2VkZ2VzIjogaW50KGNvdW50c1tsZXZlbF0pLAogICAgICAgIH0KICAgICAgICBmb3Igc3RhZ2UgaW4gc3RhZ2VfbmFtZXM6CiAgICAgICAgICAgIHJvd1tmIkVfe3N0YWdlfSJdID0gZmxvYXQoZW5lcmdpZXNbc3RhZ2VdW2xldmVsXSkKICAgICAgICAgICAgcm93W2YicF97c3RhZ2V9Il0gPSBmbG9hdChwcm9iYWJpbGl0aWVzW3N0YWdlXVtsZXZlbF0pCiAgICAgICAgcm93cy5hcHBlbmQocm93KQogICAgcHJpbnRfbW9kZWwoc3VtbWFyeSwgcm93cykKICAgIHJldHVybiByb3dzLCBzdW1tYXJ5CgoKZGVmIHByaW50X21vZGVsKHN1bW1hcnk6IGRpY3QsIHJvd3M6IGxpc3RbZGljdF0pIC0+IE5vbmU6CiAgICBwcmludCgKICAgICAgICBmIlxue3N1bW1hcnlbJ21vZGVsJ119OiBtZXNoX3NpemU9e3N1bW1hcnlbJ21lc2hfc2l6ZSddfSAiCiAgICAgICAgZiJsZXZlbHMgTTAtTXtzdW1tYXJ5WydtZXNoX3NpemUnXX0gIgogICAgICAgIGYiKG5vIGZpbmVyIHJ1bmcgaXMgaW4gdGhpcyBjaGVja3BvaW50KSIKICAgICkKICAgIHByaW50KAogICAgICAgIGYic2hhcmVkIHN1bSBvZiBzcXVhcmVkIHdlaWdodHM6ICIKICAgICAgICBmImxpbmVhcl8wPXtzdW1tYXJ5WydzdW1fdzJfbGluZWFyXzAnXTouNGZ9ICAiCiAgICAgICAgZiJsaW5lYXJfMT17c3VtbWFyeVsnc3VtX3cyX2xpbmVhcl8xJ106LjRmfSIKICAgICkKICAgIGhlYWRlciA9IGYieydsZXZlbCc6PDZ9eydMX2ttJzo+MTB9eyduX2VkZ2VzJzo+MTB9eydFX2xpbmVhcjAnOj4xNH17J0VfbWxwJzo+MTR9eydFX2VuY29kZXInOj4xNH0iCiAgICBwcmludChoZWFkZXIpCiAgICBmb3Igcm93IGluIHJvd3M6CiAgICAgICAgcHJpbnQoCiAgICAgICAgICAgIGYiTXtyb3dbJ2xldmVsJ106PDV9e3Jvd1snTF9rbSddOjEwLjFmfXtyb3dbJ25fZWRnZXMnXToxMGR9IgogICAgICAgICAgICBmIntyb3dbJ0VfbGluZWFyXzAnXToxNC42ZX17cm93WydFX21scCddOjE0LjZlfXtyb3dbJ0VfZW5jb2RlciddOjE0LjZlfSIKICAgICAgICApCiAgICBmb3Igc3RhZ2UgaW4gKCJsaW5lYXJfMCIsICJtbHAiLCAiZW5jb2RlciIpOgogICAgICAgIHByaW50KAogICAgICAgICAgICBmIntzdGFnZTo8MTB9IEg9e3N1bW1hcnlbZidIX3tzdGFnZX0nXTouNGZ9IG5hdHMgICIKICAgICAgICAgICAgZiJIX249e3N1bW1hcnlbZidIbl97c3RhZ2V9J106LjRmfSAgIgogICAgICAgICAgICBmIkUoayk9IHtzdW1tYXJ5W2YncHJlZmFjdG9yX3tzdGFnZX0nXTouNGV9IGtee3N1bW1hcnlbZidzbG9wZV97c3RhZ2V9J106LjRmfSAgIgogICAgICAgICAgICBmIlIyPXtzdW1tYXJ5W2YncjJfe3N0YWdlfSddOi40Zn0gICIKICAgICAgICAgICAgZiJNMD17MTAwICogc3VtbWFyeVtmJ20wX3tzdGFnZX0nXTouMWZ9JSIKICAgICAgICApCgoKZGVmIHdyaXRlX3RhYmxlcyhyb3dzOiBsaXN0W2RpY3RdLCBzdW1tYXJpZXM6IGxpc3RbZGljdF0sIGRlc3RpbmF0aW9uOiBQYXRoKSAtPiBOb25lOgogICAgbGV2ZWxfcGF0aCA9IGRlc3RpbmF0aW9uIC8gImdyYXBoY2FzdF93ZWlnaHRfZW50cm9weS5jc3YiCiAgICB3aXRoIGxldmVsX3BhdGgub3BlbigidyIsIG5ld2xpbmU9IiIsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIGhhbmRsZToKICAgICAgICB3cml0ZXIgPSBjc3YuRGljdFdyaXRlcihoYW5kbGUsIGZpZWxkbmFtZXM9bGlzdChyb3dzWzBdLmtleXMoKSkpCiAgICAgICAgd3JpdGVyLndyaXRlaGVhZGVyKCkKICAgICAgICB3cml0ZXIud3JpdGVyb3dzKHJvd3MpCiAgICBzdW1tYXJ5X3BhdGggPSBkZXN0aW5hdGlvbiAvICJncmFwaGNhc3Rfd2VpZ2h0X2VudHJvcHlfc3VtbWFyeS5jc3YiCiAgICB3aXRoIHN1bW1hcnlfcGF0aC5vcGVuKCJ3IiwgbmV3bGluZT0iIiwgZW5jb2Rpbmc9InV0Zi04IikgYXMgaGFuZGxlOgogICAgICAgIHdyaXRlciA9IGNzdi5EaWN0V3JpdGVyKGhhbmRsZSwgZmllbGRuYW1lcz1saXN0KHN1bW1hcmllc1swXS5rZXlzKCkpKQogICAgICAgIHdyaXRlci53cml0ZWhlYWRlcigpCiAgICAgICAgd3JpdGVyLndyaXRlcm93cyhzdW1tYXJpZXMpCiAgICBwcmludChmIlxud3JvdGUge2xldmVsX3BhdGh9IikKICAgIHByaW50KGYid3JvdGUge3N1bW1hcnlfcGF0aH0iKQoKCmRlZiB3cml0ZV9maWd1cmUocm93czogbGlzdFtkaWN0XSwgZGVzdGluYXRpb246IFBhdGgpIC0+IE5vbmU6CiAgICBtb2RlbHMgPSBbXQogICAgZm9yIHJvdyBpbiByb3dzOgogICAgICAgIGlmIHJvd1sibW9kZWwiXSBub3QgaW4gbW9kZWxzOgogICAgICAgICAgICBtb2RlbHMuYXBwZW5kKHJvd1sibW9kZWwiXSkKICAgIGZpZ3VyZSwgYXhlcyA9IHBsdC5zdWJwbG90cygxLCBsZW4obW9kZWxzKSwgZmlnc2l6ZT0oNC40ICogbGVuKG1vZGVscyksIDQuMiksIHNoYXJleT1GYWxzZSkKICAgIGlmIGxlbihtb2RlbHMpID09IDE6CiAgICAgICAgYXhlcyA9IFtheGVzXQogICAgc3R5bGVzID0gKAogICAgICAgICgiRV9saW5lYXJfMCIsICJvIiwgImxpbmVhcl8wIG9ubHkiKSwKICAgICAgICAoIkVfbWxwIiwgInMiLCAiYm90aCBsaW5lYXIgbWFwcyIpLAogICAgICAgICgiRV9lbmNvZGVyIiwgIkQiLCAiYWZ0ZXIgbGF5ZXIgbm9ybSIpLAogICAgKQogICAgZm9yIGF4aXMsIG1vZGVsIGluIHppcChheGVzLCBtb2RlbHMpOgogICAgICAgIG1vZGVsX3Jvd3MgPSBbcm93IGZvciByb3cgaW4gcm93cyBpZiByb3dbIm1vZGVsIl0gPT0gbW9kZWxdCiAgICAgICAgd2F2ZW51bWJlciA9IG5wLmFycmF5KFtyb3dbImtfcGVyX2ttIl0gZm9yIHJvdyBpbiBtb2RlbF9yb3dzXSkKICAgICAgICBmb3Iga2V5LCBtYXJrZXIsIGxhYmVsIGluIHN0eWxlczoKICAgICAgICAgICAgZW5lcmd5ID0gbnAuYXJyYXkoW3Jvd1trZXldIGZvciByb3cgaW4gbW9kZWxfcm93c10pCiAgICAgICAgICAgIGF4aXMubG9nbG9nKHdhdmVudW1iZXIsIGVuZXJneSwgbWFya2VyPW1hcmtlciwgbGFiZWw9bGFiZWwpCiAgICAgICAgcmVmZXJlbmNlID0gbnAuYXJyYXkoW3Jvd1siRV9saW5lYXJfMCJdIGZvciByb3cgaW4gbW9kZWxfcm93c10pCiAgICAgICAga29sbW9nb3JvdiA9IHJlZmVyZW5jZVswXSAqICh3YXZlbnVtYmVyIC8gd2F2ZW51bWJlclswXSkgKiogKC01LjAgLyAzLjApCiAgICAgICAgYXhpcy5sb2dsb2cod2F2ZW51bWJlciwga29sbW9nb3JvdiwgIi0tIiwgY29sb3I9IjAuNCIsIGxhYmVsPXIiayReey01LzN9JCIpCiAgICAgICAgYXhpcy5zZXRfdGl0bGUobW9kZWwpCiAgICAgICAgYXhpcy5zZXRfeGxhYmVsKHIiayA9IDEvTCAgKGttJF57LTF9JCkiKQogICAgICAgIGF4aXMuc2V0X3lsYWJlbChyIkUoaykiKQogICAgICAgIGF4aXMuZ3JpZChUcnVlLCB3aGljaD0iYm90aCIsIGFscGhhPTAuMykKICAgIGF4ZXNbMF0ubGVnZW5kKGZvbnRzaXplPTgpCiAgICBmaWd1cmUudGlnaHRfbGF5b3V0KCkKICAgIHBhdGggPSBkZXN0aW5hdGlvbiAvICJncmFwaGNhc3Rfd2VpZ2h0X2VudHJvcHkucG5nIgogICAgZmlndXJlLnNhdmVmaWcocGF0aCwgZHBpPTE0MCkKICAgIHBsdC5jbG9zZShmaWd1cmUpCiAgICBwcmludChmIndyb3RlIHtwYXRofSIpCgoKZGVmIG1haW4oKSAtPiBOb25lOgogICAgbWVzaF9tb2R1bGUgPSBsb2FkX21lc2goKQogICAgcm93czogbGlzdFtkaWN0XSA9IFtdCiAgICBzdW1tYXJpZXM6IGxpc3RbZGljdF0gPSBbXQogICAgZm9yIG5hbWUsIGZpbGVuYW1lIGluIE1PREVMUzoKICAgICAgICBtb2RlbF9yb3dzLCBzdW1tYXJ5ID0gYW5hbHl6ZShuYW1lLCBmaWxlbmFtZSwgbWVzaF9tb2R1bGUpCiAgICAgICAgcm93cy5leHRlbmQobW9kZWxfcm93cykKICAgICAgICBzdW1tYXJpZXMuYXBwZW5kKHN1bW1hcnkpCiAgICBkZXN0aW5hdGlvbiA9IG91dHB1dF9kaXIoKQogICAgd3JpdGVfdGFibGVzKHJvd3MsIHN1bW1hcmllcywgZGVzdGluYXRpb24pCiAgICB3cml0ZV9maWd1cmUocm93cywgZGVzdGluYXRpb24pCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo='}
root = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path('.')
for name, encoded in PAYLOAD.items():
    path = root / name
    path.write_bytes(base64.b64decode(encoded))
    print('wrote', path)
import runpy
runpy.run_path(str(root / 'weight_entropy.py'), run_name='__main__')
